# Deploy the GraphRAG Agent to AgentCore Runtime

> **Hosted Workshop Note:** During hosted OneBlink workshops, SageMaker roles do not have the IAM permissions required to run this notebook. You can review the code and concepts here, but deployment will only work in your own AWS account with the appropriate permissions configured.

In `01_strands_graphrag_agent.ipynb` you built and tested a Strands GraphRAG agent locally. This notebook deploys that **same agent** as a managed service on Amazon Bedrock AgentCore Runtime, then invokes it over REST. Every attendee ends up with their own deployed agent ARN.

**Architecture:**
```
User → AgentCore Runtime → Strands Agent (Claude) → semantic_search / graph_enriched_search → Neo4j
```

The agent code and dependencies are pre-built in the `agentcore_deploy/` directory:
- `agent.py` — a `BedrockAgentCoreApp` handler that wraps the retrievers and tools imported from `graphrag_agent.py`
- `graphrag_agent.py` — the reusable retrievers and `@tool` functions (canonical copy in `lib/`, copied in at deploy time)
- `pyproject.toml` — Python dependencies, including `neo4j-graphrag[bedrock]` for the retrievers and Titan embedder

You'll learn how to:
- Fill the deployment package with your Neo4j credentials from `CONFIG.txt`
- Deploy using `direct_code_deploy` (no Docker required)
- Invoke the deployed agent via the CLI and boto3

**Prerequisites:** Complete `01_strands_graphrag_agent.ipynb` first. Ensure `../CONFIG.txt` is configured and the Lab 1 seed load populated the graph with chunks, embeddings, and the `chunkEmbeddings` vector index.

## 1. Setup

Install the AgentCore starter toolkit and dependencies, then verify your SageMaker role has the required deployment permissions.

In [ ]:
%pip install bedrock-agentcore-starter-toolkit>=0.3.3 bedrock-agentcore>=1.4.7 pyyaml python-dotenv -q

In [ ]:
import boto3
from botocore.exceptions import ClientError

# Detect current SageMaker execution role
sts = boto3.client("sts")
caller = sts.get_caller_identity()
account_id = caller["Account"]
role_name = caller["Arn"].split("/")[1]

# Check if the deployment policy is attached
POLICY_NAME = "BedrockAgentCoreLabDeployPolicy"
POLICY_ARN = f"arn:aws:iam::{account_id}:policy/{POLICY_NAME}"

iam = boto3.client("iam")
try:
    attached = iam.list_attached_role_policies(RoleName=role_name)
except ClientError as e:
    # Hosted workshop roles typically lack iam:ListAttachedRolePolicies, so we
    # can't verify the policy from here. Warn and continue rather than block
    # people running this in their own accounts.
    error_code = e.response["Error"]["Code"]
    print(f"Could not verify role policies ({error_code}).")
    print("If deployment later fails with a permissions error, ask your")
    print("workshop admin to run:  ./setup/grant_sagemaker_access.sh")
else:
    policy_attached = any(
        p["PolicyArn"] == POLICY_ARN for p in attached["AttachedPolicies"]
    )
    if policy_attached:
        print(f"Deployment policy is attached to {role_name}")
    else:
        print(f"ERROR: {POLICY_NAME} is not attached to your role: {role_name}")
        print()
        print("Ask your workshop admin to run:  ./setup/grant_sagemaker_access.sh")
        print("Then re-run this cell.")
        raise SystemExit("Deployment policy not attached")

## 2. Fill agent.py from CONFIG.txt

The `agent.py` template ships with placeholder constants for the Neo4j connection. Read your credentials from `../CONFIG.txt` and write them into `agent.py`, so you never retype the values you entered back in Lab 1. The deployed agent carries these values (acceptable for the ephemeral workshop Aura instance).

This cell is idempotent: it rewrites the constant lines in place, so you can safely re-run it after editing `CONFIG.txt`.

> **Note:** The deployed `agent.py` will contain your Neo4j password in plain text. The repo copy stays as a template with placeholder tokens; this cell rewrites it in place with your real values. Do not commit the filled copy back. The generated `.bedrock_agentcore.yaml` is already git-ignored.

In [ ]:
import re
import shutil
from pathlib import Path

from dotenv import dotenv_values

config = dotenv_values("../CONFIG.txt")

DEFAULTS = {
    "MODEL_ID": "us.anthropic.claude-sonnet-4-6",
    "REGION": "us-east-1",
}
KEYS = ["NEO4J_URI", "NEO4J_USERNAME", "NEO4J_PASSWORD", "MODEL_ID", "REGION"]

values = {}
for key in KEYS:
    val = config.get(key) or DEFAULTS.get(key)
    if not val:
        raise ValueError(f"{key} is missing from ../CONFIG.txt")
    values[key] = val

agent_path = Path("agentcore_deploy/agent.py")
source = agent_path.read_text()

# Replace each `KEY = ...` assignment line, whether it currently holds a
# placeholder token or a previously written value. This keeps the cell re-runnable.
for key, val in values.items():
    source = re.sub(
        rf"^{key} = .*$",
        f"{key} = {val!r}",
        source,
        count=1,
        flags=re.MULTILINE,
    )

agent_path.write_text(source)

# agent.py imports the reusable retrievers/tools from graphrag_agent.py. That
# module's canonical copy lives in lib/; copy it into the deploy directory so
# direct_code_deploy (which bundles only agentcore_deploy/) can import it. This
# copy is git-ignored.
shutil.copy(Path("lib/graphrag_agent.py"), Path("agentcore_deploy/graphrag_agent.py"))

print("agent.py updated from CONFIG.txt:")
print(f"  NEO4J_URI:      {values['NEO4J_URI']}")
print(f"  NEO4J_USERNAME: {values['NEO4J_USERNAME']}")
print(f"  NEO4J_PASSWORD: {'*' * 8} (hidden)")
print(f"  MODEL_ID:       {values['MODEL_ID']}")
print(f"  REGION:         {values['REGION']}")
print("Copied lib/graphrag_agent.py -> agentcore_deploy/graphrag_agent.py")

## 3. Review the Deployment Package

Confirm the package contents and inspect the filled `agent.py` so you can see the GraphRAG tools that will be deployed.

In [ ]:
!ls -la agentcore_deploy/

In [ ]:
!cat agentcore_deploy/agent.py

## 4. Configure and Deploy

AgentCore needs a `.bedrock_agentcore.yaml` config that specifies the entrypoint, deployment type, and AWS settings. Generate it programmatically to avoid interactive CLI prompts. `execution_role_auto_create: True` lets the toolkit create the runtime execution role for you.

The toolkit packages your code locally and cross-compiles the dependencies for Linux ARM64 using prebuilt wheels, so no Docker is required. `neo4j-graphrag` ships as a pure-Python wheel on PyPI, which installs cleanly under this cross-compilation.

In [ ]:
import os

import boto3
import yaml
from dotenv import load_dotenv

load_dotenv("../CONFIG.txt")

REGION = os.getenv("REGION", "us-east-1")

sts = boto3.client("sts")
account_id = sts.get_caller_identity()["Account"]

agent_dir = os.path.abspath("agentcore_deploy")
entrypoint = os.path.join(agent_dir, "agent.py")

AGENT_NAME = "graphrag_strands_agent"

config = {
    "default_agent": AGENT_NAME,
    "agents": {
        AGENT_NAME: {
            "name": AGENT_NAME,
            "language": "python",
            "entrypoint": entrypoint,
            "deployment_type": "direct_code_deploy",
            "runtime_type": "PYTHON_3_13",
            "platform": "linux/arm64",
            "source_path": agent_dir,
            "aws": {
                "account": account_id,
                "region": REGION,
                "execution_role_auto_create": True,
                "ecr_auto_create": False,
                "s3_auto_create": True,
                "network_configuration": {
                    "network_mode": "PUBLIC",
                },
                "protocol_configuration": {
                    "server_protocol": "HTTP",
                },
                "observability": {
                    "enabled": True,
                },
            },
        }
    },
}

config_path = os.path.join(agent_dir, ".bedrock_agentcore.yaml")
with open(config_path, "w") as f:
    yaml.dump(config, f, default_flow_style=False)

print(f"Wrote {config_path}")
print(f"  Agent:      {AGENT_NAME}")
print(f"  Account:    {account_id}")
print(f"  Region:     {REGION}")
print(f"  Deploy:     direct_code_deploy")

In [ ]:
# Install zip if not available (needed by direct_code_deploy)
!which zip || (sudo apt-get update -qq && sudo apt-get install -y -qq zip)

# Deploy (--auto-update-on-conflict updates the agent if it already exists)
!cd agentcore_deploy && agentcore deploy --auto-update-on-conflict

## 5. Invoke the Deployed Agent

Once deployed, invoke the agent via the `agentcore` CLI or programmatically with boto3. The hero question is entity-specific, so the agent picks `graph_enriched_search` and traverses NVIDIA's products and risk factors.

In [ ]:
# Invoke via CLI
!cd agentcore_deploy && agentcore invoke '{"prompt": "What are NVIDIA'"'"'s biggest risk factors, and which of its products are most exposed?"}'

### Invoke via boto3

For programmatic access, use the `bedrock-agentcore` boto3 client. First read the deployed agent ARN from the generated config.

In [ ]:
import yaml

with open("agentcore_deploy/.bedrock_agentcore.yaml") as f:
    deploy_config = yaml.safe_load(f)

default_agent = deploy_config["default_agent"]
agent_config = deploy_config["agents"][default_agent]
AGENT_ARN = agent_config["bedrock_agentcore"]["agent_arn"]
AGENT_REGION = agent_config["aws"]["region"]

print(f"Agent:  {default_agent}")
print(f"ARN:    {AGENT_ARN}")
print(f"Region: {AGENT_REGION}")

In [ ]:
import json
import uuid

from botocore.config import Config

agentcore_config = Config(read_timeout=300)


def invoke_agent(prompt: str):
    """Invoke the deployed agent via boto3."""
    client = boto3.client(
        "bedrock-agentcore",
        region_name=AGENT_REGION,
        config=agentcore_config,
    )

    response = client.invoke_agent_runtime(
        agentRuntimeArn=AGENT_ARN,
        runtimeSessionId=str(uuid.uuid4()),
        payload=json.dumps({"prompt": prompt}).encode(),
        qualifier="DEFAULT",
    )

    content = "".join(
        chunk.decode("utf-8") for chunk in response.get("response", [])
    )

    try:
        return json.loads(content)
    except (json.JSONDecodeError, ValueError):
        return content


result = invoke_agent(
    "What are NVIDIA's biggest risk factors, and which of its products are most exposed?"
)
print(result)

## 6. Cleanup

When you're done, remove the agent from AgentCore Runtime. This deletes the deployed runtime but keeps your local files.

In [ ]:
# Uncomment the line below to destroy the deployed agent
# !cd agentcore_deploy && agentcore destroy